# Modir — Whisper Lebanese-Arabic ASR fine-tune (Colab GPU runner)

**This notebook holds NO training logic (AD-12.1).** The pipeline is the
version-controlled code under `backend/app/asr/*.py` — this notebook only
pip-installs the pinned `asr` deps, imports `app.asr`, runs the training on a
Colab T4/A100, and uploads the produced artifact. If you find yourself writing
training code here, move it into `app/asr/` instead.

Runtime → Change runtime type → **GPU** before running.

## 1. Clone the repo and install the pinned ASR extra

The versions are the single source of truth in `backend/pyproject.toml`
(`[project.optional-dependencies] asr`). Installing the extra pulls the heavy
torch/transformers/datasets stack ONLY here, never on the CI/dev path (AD-12.6).

In [ ]:
# Replace with your repo URL / branch when running on Colab.
!git clone https://github.com/ali-hamad0/MOUDIR.git
%cd MOUDIR/backend
# Install the pinned ASR extra (torch, transformers, datasets, evaluate, jiwer,
# soundfile, accelerate) defined in pyproject.toml. pip resolves the same pins uv
# locks; on Colab the CUDA torch wheel is selected automatically.
!pip install -q -e '.[asr]'

## 2. Run the fine-tune (imports `app.asr.train`)

One command — the SAME entrypoint you run locally for a shape check. The full GPU
run writes the model + processor to `app/asr/artifacts/` and a `model_card.json`,
and appends the zero-shot baseline + fine-tuned WER/CER rows to `app/asr/results.csv`.

In [ ]:
# Task 12.3 fills app.asr.train.train(); until then this raises NotImplementedError.
!python -m app.asr.train

## 3. Upload the artifact (it is git-ignored — AD-12.4)

The fine-tuned whisper-small is ~1 GB and is NEVER committed. Push it to the chosen
out-of-repo home (a GitHub Release asset, a Hugging Face repo, or the project MinIO
bucket) and record the URI in `WHISPER_MODEL_URI`. The serving app fetches it to
`whisper_model_path` before `transcribe_mode="whisper"` is used.

In [ ]:
# Artifact lives at app/asr/artifacts/<name>/ . Upload step wired in Task 12.3/12.7
# (HF Hub push, Release asset, or MinIO put). The model_card.json is committed; the
# weights are not.
!ls -la app/asr/artifacts/